## Import

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
import numpy as np
import scipy.linalg as linalg
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
#import preprocess


device = 'cuda:4'

%load_ext autoreload
%autoreload 2

## Data generation

In [2]:
from datasets import NonlinearGaussian, MoG

n, d = 10000, 64                                 # < higher d, higher MI
true_rho = 0.7                                   # < higher rho, higher MI
case = '3a'                                      # < choose between ['1a', '1b', '2', '3a', '3b', '3c', 'MoG']

if case != 'MoG':
    dataset = NonlinearGaussian.NonlinearGaussian(n_samples=n, n_dims=d, rho=true_rho, mu=0, case=case)
    X0, Y0 = dataset.sample_data(n_samples = n)
    X, Y = dataset.transformation(X0, Y0)
    MI = dataset.true_mutual_info()              # we know GT MI
else:
    dataset = MoG.MoG(n_samples=n, n_dims=d, K=5, shifts=[-0.4, -0.1, 0, 0.1, 0.4], rhos=[0.5, 0.6, 0.7, 0.8, 0.9])
    X, Y = dataset.sample_data(n_samples = n)
    MI = dataset.empirical_mutual_info()         # MI by MC estimate

X, Y = X.to(device), Y.to(device)
Z = torch.cat([X, Y], dim=1)
T = torch.ones(n, 2).to(device)

print('X size=', X.size(), 'Y size=', Y.size())
print("True MI is", MI)

X size= torch.Size([10000, 32]) Y size= torch.Size([10000, 32])
True MI is 10.773512852220248


## MI estimation

In [4]:
class Hyperparams(object):
    def __init__(self): 
        self.critic = 'neural'                # ('neural', 'quadratic')
        self.lr = 5e-4
        self.bs = 500
        self.n_bridges = 4
        self.wd = 1e-5
        self.max_iteration = 1250
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]

In [16]:
## Mutual information neural diffusion estimate (MINDE)
from estimators.MINDE import MINDE

hyperparams.t_patience = 500
hyperparams.dim = d//2
hyperparams.device = device
hyperparams.importance_sampling = True

estimator = MINDE(None, None, None, hyperparams)

estimator.to(device)
estimator.learn(X, Y)

print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.981761634349823 loss val= 0.9949260950088501 best val loss= 0.9949260950088501 best t= 0
finished: t= 63 loss= 0.3081326186656952 loss val= 0.33337631821632385 best val loss= 0.25029075145721436 best t= 60
finished: t= 126 loss= 0.30068105459213257 loss val= 0.30366766452789307 best val loss= 0.24416086077690125 best t= 75
finished: t= 189 loss= 0.2808900773525238 loss val= 0.28865641355514526 best val loss= 0.2141769379377365 best t= 136
finished: t= 252 loss= 0.19234569370746613 loss val= 0.2494477927684784 best val loss= 0.20602208375930786 best t= 239
finished: t= 315 loss= 0.2493913322687149 loss val= 0.23071013391017914 best val loss= 0.20369279384613037 best t= 259
finished: t= 378 loss= 0.24583378434181213 loss val= 0.28128862380981445 best val loss= 0.20369279384613037 best t= 259
finished: t= 441 loss= 0.25808659195899963 loss val= 0.2792362868785858 best val loss= 0.1857892870903015 best t= 393
finished: t= 504 loss= 0.2268057018518448 loss val= 0.2320

In [15]:
## Mutual information neural estimate (MINE)
from estimators.MINE import MINE

estimator = MINE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)


print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.011183977127075195 loss val= 0.01203867793083191 best val loss= 0.01203867793083191 best t= 0
finished: t= 63 loss= -1.6538426876068115 loss val= -1.599865436553955 best val loss= -1.6144390106201172 best t= 61
finished: t= 126 loss= -2.217897415161133 loss val= -2.0509891510009766 best val loss= -2.1109766960144043 best t= 123
finished: t= 189 loss= -2.452944278717041 loss val= -2.194395065307617 best val loss= -2.2884345054626465 best t= 171
finished: t= 252 loss= -2.333411693572998 loss val= -2.299561023712158 best val loss= -2.4967458248138428 best t= 248
finished: t= 315 loss= -2.642728805541992 loss val= -2.4223289489746094 best val loss= -2.6146740913391113 best t= 296
finished: t= 378 loss= -3.0947370529174805 loss val= -2.6758575439453125 best val loss= -2.749878406524658 best t= 365
finished: t= 441 loss= -3.1552464962005615 loss val= -2.7609879970550537 best val loss= -2.7936649322509766 best t= 424
finished: t= 504 loss= -2.7979068756103516 loss val= 

In [14]:
## Neural adaptive MI estimate
from estimators.VCE import VCE

estimator = VCE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

K components= 5 copula transform= True
nde type: FM
finished: t= 0 loss= 1.034378170967102 loss val= 1.0510226488113403 best val loss= 1.0510226488113403 best t= 0
finished: t= 126 loss= 0.21394912898540497 loss val= 0.19877848029136658 best val loss= 0.1902483105659485 best t= 112
finished: t= 252 loss= 0.19818095862865448 loss val= 0.19429965317249298 best val loss= 0.1872175931930542 best t= 167
finished: t= 378 loss= 0.19374167919158936 loss val= 0.19273430109024048 best val loss= 0.18582406640052795 best t= 345
finished: t= 504 loss= 0.2016821801662445 loss val= 0.1821439415216446 best val loss= 0.1796899139881134 best t= 489
finished: t= 630 loss= 0.1649922877550125 loss val= 0.18549519777297974 best val loss= 0.1789480596780777 best t= 540
finished: t= 756 loss= 0.1901082843542099 loss val= 0.18807968497276306 best val loss= 0.17681370675563812 best t= 669
finished: t= 882 loss= 0.20158393681049347 loss val= 0.1893833428621292 best val loss= 0.1749197095632553 best t= 861
finish